# Aggregate PathoCellBench Results

This notebook aggregates results from multiple seeds for PathoCellBench evaluation.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# Get paths from snakemake
result_files = [Path(f) for f in snakemake.input.results]
output_summary = Path(snakemake.output.summary)

logger.info(f"Aggregating {len(result_files)} result files")
for f in result_files:
    logger.info(f"  - {f}")

In [ ]:
# Load all results
results = []
for result_file in result_files:
    with open(result_file, 'r') as f:
        result = json.load(f)
        results.append(result)

logger.info(f"Loaded {len(results)} results")

In [ ]:
# Extract metric keys (excluding non-metric fields)
non_metric_keys = {'n_cells', 'n_patches', 'n_observations', 'n_cell_types', 'cell_types', 'seed', 'prediction_level'}
metric_keys = [k for k in results[0].keys() if k not in non_metric_keys]

logger.info(f"Found {len(metric_keys)} metrics to aggregate")

In [ ]:
# Compute mean and std for each metric
summary = {
    'n_seeds': len(results),
    'seeds': [r['seed'] for r in results],
    'n_observations': results[0].get('n_observations', results[0].get('n_cells', results[0].get('n_patches', 0))),
    'n_cell_types': results[0]['n_cell_types'],
    'cell_types': results[0]['cell_types'],
}

for metric in metric_keys:
    values = [r[metric] for r in results]
    summary[f"{metric}_mean"] = float(np.mean(values))
    summary[f"{metric}_std"] = float(np.std(values))
    summary[f"{metric}_values"] = values

# Add metrics with names expected by spider plot
# Map our cell type classification metrics to spider plot metric names
if 'f1_macroAvg_mean' in summary:
    # Use F1 as the main metric for zero-shot classification
    summary['pathocell_zero_shot_classification'] = summary['f1_macroAvg_mean']
    summary['pathocell_image_text_retrieval'] = summary.get('recall_at_1_macroAvg_mean', summary['f1_macroAvg_mean'])
    summary['pathocell_embedding_quality'] = summary.get('rocauc_macroAvg_mean', summary['f1_macroAvg_mean'])

# Display key metrics
logger.info("\n=== Aggregated Results ===")
logger.info(f"Accuracy (macro): {summary.get('accuracy_macroAvg_mean', 0):.4f} ± {summary.get('accuracy_macroAvg_std', 0):.4f}")
logger.info(f"F1 score (macro): {summary.get('f1_macroAvg_mean', 0):.4f} ± {summary.get('f1_macroAvg_std', 0):.4f}")
logger.info(f"Precision (macro): {summary.get('precision_macroAvg_mean', 0):.4f} ± {summary.get('precision_macroAvg_std', 0):.4f}")
logger.info(f"AUROC (macro): {summary.get('rocauc_macroAvg_mean', 0):.4f} ± {summary.get('rocauc_macroAvg_std', 0):.4f}")
logger.info(f"\nSpider plot metrics:")
logger.info(f"  pathocell_zero_shot_classification: {summary.get('pathocell_zero_shot_classification', 0):.4f}")
logger.info(f"  pathocell_image_text_retrieval: {summary.get('pathocell_image_text_retrieval', 0):.4f}")
logger.info(f"  pathocell_embedding_quality: {summary.get('pathocell_embedding_quality', 0):.4f}")

In [ ]:
# Save summary
output_summary.parent.mkdir(parents=True, exist_ok=True)
with open(output_summary, 'w') as f:
    json.dump(summary, f, indent=2)

logger.info(f"Saved aggregated summary to {output_summary}")
logger.info("Aggregation complete!")